# Cài thư viện + Setup

In [1]:
!pip install -q pyvi emoji transformers scikit-learn openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 58.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 49.3 MB/s eta 0:00:00


In [2]:


import os
import json
import pickle
import time
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, AutoConfig, get_scheduler
from sklearn.metrics import f1_score, classification_report
from pyvi.ViTokenizer import tokenize
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

os.makedirs("/kaggle/working/saved_models", exist_ok=True)
os.makedirs("/kaggle/working/reports", exist_ok=True)
os.makedirs("/kaggle/working/corpus", exist_ok=True)

Device: cuda


# Load dữ liệu 

In [3]:
import subprocess

REPO_URL = "https://github.com/ricardo-tran/ViGoEmotions.git"
REPO_DIR = "/kaggle/working/ViGoEmotions_Original"

if not os.path.exists(REPO_DIR):
    print("--- Đang clone repo ---")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    print("--- Repo đã tồn tại ---")

DOCS_PATH = os.path.join(REPO_DIR, "model", "docs")
CORPUS_PATH = os.path.join(REPO_DIR, "corpus")

print("DOCS_PATH  :", DOCS_PATH)
print("CORPUS_PATH:", CORPUS_PATH)

# Kiểm tra xem có file pickle S3 không
pkl_candidates = [
    os.path.join(CORPUS_PATH, "s3_visolex_train-val-test.pkl"),
    os.path.join(REPO_DIR, "corpus", "s3_visolex_train-val-test.pkl"),
]

pkl_path = None
for p in pkl_candidates:
    if os.path.exists(p):
        pkl_path = p
        break

if pkl_path:
    print(f" Tìm thấy pickle S3: {pkl_path}")
    with open(pkl_path, "rb") as f:
        train_df, val_df, test_df = pickle.load(f)
else:
    print(" Không thấy s3_visolex_train-val-test.pkl")
    print("→ Tạm dùng dataset_V1.xlsx (giống S2) + tiền xử lý S3 (chỉ lower)")
    excel_path = os.path.join(CORPUS_PATH, "dataset_V1.xlsx")
    excel_file = pd.ExcelFile(excel_path)

    if "train" in excel_file.sheet_names:
        train_df = pd.read_excel(excel_file, sheet_name="train")
        val_df   = pd.read_excel(excel_file, sheet_name="val")
        test_df  = pd.read_excel(excel_file, sheet_name="test")
    else:
        df = pd.read_excel(excel_file, sheet_name="Sheet1")
        train_df = df[df["set"] == "train"].copy()
        val_df   = df[df["set"] == "val"].copy()
        test_df  = df[df["set"] == "test"].copy()

print("Train:", train_df.shape)
print("Val  :", val_df.shape)
print("Test :", test_df.shape)
display(train_df.head(3))

--- Đang clone repo ---


Cloning into '/kaggle/working/ViGoEmotions_Original'...


DOCS_PATH  : /kaggle/working/ViGoEmotions_Original/model/docs
CORPUS_PATH: /kaggle/working/ViGoEmotions_Original/corpus
 Không thấy s3_visolex_train-val-test.pkl
→ Tạm dùng dataset_V1.xlsx (giống S2) + tiền xử lý S3 (chỉ lower)
Train: (16531, 3)
Val  : (2066, 3)
Test : (2067, 3)


,id,text,labels
0,tik000008,Xem mà ngẫm lại cuộc đời bản thân ta đã trải q...,[12]
1,5743,bức ảnh xuất sắc ❤️,"[2, 8, 3]"
2,32895,"Vừa đẹp trai, vừa tài giỏi. Nhà mặt phố, bố là...","[8, 7]"


# Tiền xử lý

In [4]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    return text.lower()

print("Applying S3 preprocessing (only lower)...")
for df in [train_df, val_df, test_df]:
    df["text"] = df["text"].astype(str).apply(clean_text)

print("Done")
print(train_df["text"].iloc[0])

Applying S3 preprocessing (only lower)...
Done
xem mà ngẫm lại cuộc đời bản thân ta đã trải qua nhiều thứ ta rồi cũng sẽ lớn kí ước sẽ còn mãi trong lòng


# Load label dict + Encode labels

In [5]:
# Ưu tiên lấy từ model/docs, nếu không có thì lấy từ corpus
label_dict_path = os.path.join(DOCS_PATH, "label_dict.json")
if not os.path.exists(label_dict_path):
    label_dict_path = os.path.join(CORPUS_PATH, "label_dict.json")

with open(label_dict_path, "r", encoding="utf-8") as f:
    label_dict = json.load(f)

label_to_idx = {label: int(idx) for idx, label in label_dict.items()}
print("Number of labels:", len(label_dict))
print("label_dict path:", label_dict_path)

def encode_labels(label, label_dict):
    labels = str(label).replace("[", "").replace("]", "").replace("'", "").replace('"', "").split(",")
    labels = [x.strip() for x in labels if x.strip()]
    vec = np.zeros(len(label_dict), dtype=np.float32)

    if labels and labels[0].isnumeric():
        labels = [int(x) for x in labels]
        for idx in label_dict.values():
            if idx in labels:
                vec[idx] = 1.0
    else:
        for lab, idx in label_dict.items():
            if lab in labels:
                vec[idx] = 1.0
    return vec

train_texts  = train_df["text"].tolist()
train_labels = [encode_labels(x, label_to_idx) for x in train_df["labels"]]
val_texts    = val_df["text"].tolist()
val_labels   = [encode_labels(x, label_to_idx) for x in val_df["labels"]]
test_texts   = test_df["text"].tolist()
test_labels  = [encode_labels(x, label_to_idx) for x in test_df["labels"]]

print("Labels encoded")

Number of labels: 28
label_dict path: /kaggle/working/ViGoEmotions_Original/model/docs/label_dict.json
Labels encoded


# Dataset + DataLoader

In [6]:
model_type = "cafe"
model_name = "uitnlp/CafeBERT"
max_len = 200
BATCH_SIZE = 16

from pyvi.ViTokenizer import tokenize
tokenizer = AutoTokenizer.from_pretrained(model_name)

class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = torch.tensor(np.array(labels), dtype=torch.float32)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        text = tokenize(text) 

        encoding = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_len,
            padding="max_length",
            return_tensors="pt",
            return_attention_mask=True,
            return_token_type_ids=False,
        )
        return {
            "text": text,
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
            "targets": self.labels[idx],
        }

train_dataset = SentimentDataset(train_texts, train_labels, tokenizer, max_len)
val_dataset   = SentimentDataset(val_texts, val_labels, tokenizer, max_len)
test_dataset  = SentimentDataset(test_texts, test_labels, tokenizer, max_len)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print("DataLoader ready (CafeBERT - S3)")
print("Train batches:", len(train_loader))

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/496 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

DataLoader ready (CafeBERT - S3)
Train batches: 1034


# model

In [7]:
class ModelSentimentClassifier(nn.Module):
    def __init__(self, n_classes, model_type="cafe"):
        super().__init__()
        if model_type == "pho":
            bert_model = "vinai/phobert-base-v2"
        elif model_type == "viso":
            bert_model = "uitnlp/visobert"
        elif model_type == "cafe":
            bert_model = "uitnlp/CafeBERT"
        elif model_type == "vi":
            bert_model = "FPTAI/vibert-base-cased"
        else:
            raise ValueError(f"Unknown model_type: {model_type}")

        config = AutoConfig.from_pretrained(
            bert_model,
            hidden_dropout_prob=0.1,
            attention_probs_dropout_prob=0.1
        )
        self.bert = AutoModel.from_pretrained(bert_model, config=config)
        self.drop = nn.Dropout(0.2)
        self.fc = nn.Linear(self.bert.config.hidden_size, n_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True
        )
        pooled = outputs.pooler_output if outputs.pooler_output is not None else outputs.last_hidden_state[:, 0, :]
        x = self.drop(pooled)
        return {"logits": self.fc(x)}

model = ModelSentimentClassifier(n_classes=len(label_dict), model_type=model_type).to(device)
print("Model loaded (CafeBERT - S3)")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: uitnlp/CafeBERT
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded (CafeBERT - S3)
Parameters: 559,919,132


# Train + Lưu metrics

In [8]:
EPOCHS = 12
optimizer = AdamW(model.parameters(), lr=5e-5)
scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=len(train_loader),
    num_training_steps=len(train_loader) * EPOCHS
)

label_counts = np.sum(train_labels, axis=0)
pos_weight = torch.tensor(
    [(len(train_labels) - c) / max(c, 1) for c in label_counts],
    dtype=torch.float32
).to(device)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

def run_epoch(model, loader, is_train=True):
    model.train() if is_train else model.eval()
    losses, all_y, all_p = [], [], []

    context = torch.enable_grad() if is_train else torch.no_grad()
    with context:
        for batch in tqdm(loader, leave=False):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            targets = batch["targets"].to(device)

            if is_train:
                optimizer.zero_grad()

            logits = model(input_ids, attention_mask)["logits"]
            loss = loss_fn(logits, targets)
            losses.append(loss.item())

            if is_train:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()

            preds = (torch.sigmoid(logits) >= 0.5).int()
            all_y.append(targets.cpu().numpy())
            all_p.append(preds.cpu().numpy())

    y = np.vstack(all_y)
    p = np.vstack(all_p)
    f1 = f1_score(y, p, average="macro", zero_division=0)
    return np.mean(losses), f1

best_f1 = 0.0
history = {"epoch": [], "train_loss": [], "train_f1": [], "val_loss": [], "val_f1": []}

print("Start training PhoBERT (S3)...")
for epoch in range(1, EPOCHS + 1):
    print(f"\n===== Epoch {epoch}/{EPOCHS} =====")
    train_loss, train_f1 = run_epoch(model, train_loader, is_train=True)
    val_loss, val_f1 = run_epoch(model, val_loader, is_train=False)

    history["epoch"].append(epoch)
    history["train_loss"].append(train_loss)
    history["train_f1"].append(train_f1)
    history["val_loss"].append(val_loss)
    history["val_f1"].append(val_f1)

    print(f"Train Loss: {train_loss:.4f} | Train Macro-F1: {train_f1:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Val   Macro-F1: {val_f1:.4f}")

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), f"/kaggle/working/saved_models/{model_type}_s3_best.pth")
        print(f"Saved best model (Val F1 = {best_f1:.4f})")

# Lưu metrics
pd.DataFrame(history).to_excel(f"/kaggle/working/reports/metrics_{model_type}_s3.xlsx", index=False)
print("\nTraining finished!")
print(f"Metrics saved: /kaggle/working/reports/metrics_{model_type}_s3.xlsx")

Start training PhoBERT (S3)...

===== Epoch 1/12 =====


Train Loss: 0.9528 | Train Macro-F1: 0.2642
Val   Loss: 0.7350 | Val   Macro-F1: 0.3898
Saved best model (Val F1 = 0.3898)

===== Epoch 2/12 =====


Train Loss: 0.7410 | Train Macro-F1: 0.3899
Val   Loss: 0.7259 | Val   Macro-F1: 0.3981
Saved best model (Val F1 = 0.3981)

===== Epoch 3/12 =====


Train Loss: 0.6178 | Train Macro-F1: 0.4503
Val   Loss: 0.6580 | Val   Macro-F1: 0.4432
Saved best model (Val F1 = 0.4432)

===== Epoch 4/12 =====


Train Loss: 0.5023 | Train Macro-F1: 0.5177
Val   Loss: 0.6692 | Val   Macro-F1: 0.4849
Saved best model (Val F1 = 0.4849)

===== Epoch 5/12 =====


...

Train Loss: 0.2208 | Train Macro-F1: 0.7363
Val   Loss: 0.8589 | Val   Macro-F1: 0.5721
Saved best model (Val F1 = 0.5721)

===== Epoch 9/12 =====


Train Loss: 0.1831 | Train Macro-F1: 0.7830
Val   Loss: 0.9133 | Val   Macro-F1: 0.5778
Saved best model (Val F1 = 0.5778)

===== Epoch 10/12 =====


Train Loss: 0.1399 | Train Macro-F1: 0.8315
Val   Loss: 0.9685 | Val   Macro-F1: 0.5899
Saved best model (Val F1 = 0.5899)

===== Epoch 11/12 =====


Train Loss: 0.1086 | Train Macro-F1: 0.8717
Val   Loss: 1.0410 | Val   Macro-F1: 0.5974
Saved best model (Val F1 = 0.5974)

===== Epoch 12/12 =====


Train Loss: 0.0868 | Train Macro-F1: 0.9010
Val   Loss: 1.0795 | Val   Macro-F1: 0.6014
Saved best model (Val F1 = 0.6014)

Training finished!
Metrics saved: /kaggle/working/reports/metrics_cafe_s3.xlsx


# Test + Lưu Classification Report

In [9]:
model.load_state_dict(torch.load(f"/kaggle/working/saved_models/{model_type}_s3_best.pth"))
model.eval()

all_targets, all_preds = [], []
with torch.no_grad():
    for batch in tqdm(test_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        targets = batch["targets"].to(device)

        logits = model(input_ids, attention_mask)["logits"]
        preds = (torch.sigmoid(logits) >= 0.5).int()

        all_targets.append(targets.cpu().numpy())
        all_preds.append(preds.cpu().numpy())

y_true = np.vstack(all_targets)
y_pred = np.vstack(all_preds)

macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
micro_f1 = f1_score(y_true, y_pred, average="micro", zero_division=0)

print("\n========== TEST RESULTS (PhoBERT - S3) ==========")
print(f"Macro F1: {macro_f1:.4f}")
print(f"Micro F1: {micro_f1:.4f}")

report = classification_report(
    y_true, y_pred,
    target_names=list(label_dict.values()),
    zero_division=0,
    output_dict=True
)
pd.DataFrame(report).transpose().to_excel(
    f"/kaggle/working/reports/classification_report_{model_type}_s3.xlsx",
    index=True
)

print(classification_report(y_true, y_pred, target_names=list(label_dict.values()), zero_division=0))
print(f"\n Report saved: /kaggle/working/reports/classification_report_{model_type}_s3.xlsx")

100%|██████████| 130/130 [01:07<00:00,  1.92it/s]


========== TEST RESULTS (PhoBERT - S3) ==========
Macro F1: 0.6165
Micro F1: 0.6281
                precision    recall  f1-score   support

     amusement       0.67      0.85      0.75       374
    excitement       0.50      0.61      0.55        98
           joy       0.50      0.73      0.59       204
          love       0.61      0.86      0.72       143
        desire       0.43      0.68      0.52        80
      optimism       0.61      0.85      0.71       142
        caring       0.56      0.78      0.65       150
         pride       0.69      0.74      0.72        86
    admiration       0.49      0.62      0.55       101
     gratitude       0.82      0.91      0.86       108
        relief       0.54      0.78      0.64        60
      approval       0.56      0.72      0.63       115
   realization       0.42      0.52      0.46        95
      surprise       0.54      0.66      0.59        85
     curiosity       0.57      0.68      0.62       100
     confusion    